# Chunking Explorer — Interactive Visualization

**AI-2 Session 2.2 — Companion Notebook**

---

Chunking is one of those decisions that sounds simple ("just split the text") but has real downstream consequences for retrieval quality. This notebook lets you **see** how different chunking strategies split text and where overlaps land.

Instead of reading chunk statistics in a table, you will:

- Watch text get **color-coded** by chunk assignment
- See **overlap zones** rendered as striped regions where two chunks share the same text
- Compare **fixed-size** vs **paragraph-aware** strategies side by side on the same document
- Explore how **chunk_size** and **overlap** parameters change the result in real time

No API key required. No embeddings. Just the chunker and your eyes.

In [ ]:
import sys
from pathlib import Path

_here = Path(".").resolve()
_root = _here
while _root != _root.parent:
    if (_root / "pyproject.toml").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root))

from src.s3_ingestion.chunker import chunk_document, chunk_fixed, Chunk, count_tokens

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import plotly.graph_objects as go

print("Ready. No API key needed for this notebook.")

In [ ]:
data_dir = _root / "data" / "northbrook"

documents = {}

# Short excerpt -- great for detailed exploration
documents["Recharge Days (~140 tokens)"] = (
    "Northbrook Partners is introducing Recharge Days as a new benefit for 2025. "
    "In addition to standard PTO, all employees receive 2 Recharge Days per "
    "calendar year. Recharge Days are company-encouraged mental health days that "
    "require no justification or advance notice. Employees simply notify their "
    "manager that they are taking a Recharge Day."
    "\n\n"
    "Recharge Days do not carry over and must be used within the calendar year. "
    "The intent of Recharge Days is to normalize proactive rest. We encourage "
    "all employees to take advantage of this benefit."
    "\n\n"
    "If an employee does not use their Recharge Days by December 31, they are "
    "forfeited. Recharge Days cannot be converted to cash or combined with "
    "other leave types."
)

# Medium document
vp = data_dir / "vacation_policy_2025.md"
if vp.exists():
    documents["Vacation Policy"] = vp.read_text()

# Long document (truncated for visualization)
hb = data_dir / "employee_handbook.md"
if hb.exists():
    full = hb.read_text()
    documents["Employee Handbook (first ~750 tokens)"] = full[:3000]

print("Loaded documents:")
for name, text in documents.items():
    para_breaks = text.count("\n\n")
    print(f"  {name}: {len(text):,} chars ({count_tokens(text):,} tokens), {para_breaks} paragraph breaks")

In [ ]:
# ── Color palette ──────────────────────────────────────────────────────
COLORS = [
    "#FFE0B2", "#B3E5FC", "#C8E6C9", "#F8BBD0", "#D1C4E9",
    "#FFCCBC", "#B2DFDB", "#FFF9C4", "#CFD8DC", "#E1BEE7",
    "#FFECB3", "#B2EBF2", "#DCEDC8", "#F0F4C3", "#D7CCC8",
]
BORDERS = [
    "#FF9800", "#03A9F4", "#4CAF50", "#E91E63", "#673AB7",
    "#FF5722", "#009688", "#FFC107", "#607D8B", "#9C27B0",
    "#FFB300", "#00BCD4", "#8BC34A", "#CDDC39", "#795548",
]

import tiktoken
_enc = tiktoken.get_encoding("cl100k_base")


# ── Token-to-character mapping ─────────────────────────────────────────

def _token_char_boundaries(text):
    """Map token indices to character positions in the original text.

    Returns (tokens, boundaries) where boundaries[i] is the character
    offset where token i starts, and boundaries[len(tokens)] = len(text).
    """
    tokens = _enc.encode(text)
    boundaries = [0]
    for t in tokens:
        boundaries.append(boundaries[-1] + len(_enc.decode([t])))
    return tokens, boundaries


# ── Range computation (token-based) ───────────────────────────────────

def compute_fixed_ranges(text, chunk_size, overlap):
    """Token-based fixed-size ranges mapped to character positions."""
    tokens, boundaries = _token_char_boundaries(text)
    n = len(tokens)
    ranges = []
    start = 0
    idx = 0
    stride = max(chunk_size - overlap, 1)
    while start < n:
        end = min(start + chunk_size, n)
        ranges.append((boundaries[start], boundaries[end], idx))
        start += stride
        idx += 1
    return ranges


def compute_paragraph_ranges(text, chunk_size, overlap):
    """Token-based paragraph-aware ranges mapped to character positions."""
    # Get raw (non-overlapping) chunks — their text is a substring of the original
    raw_chunks = chunk_document(text, source="demo", chunk_size=chunk_size, overlap=0)
    if not raw_chunks:
        return []

    # Locate each raw chunk in the original text
    raw_ranges = []
    search_from = 0
    for chunk in raw_chunks:
        pos = text.find(chunk.text, search_from)
        if pos >= 0:
            raw_ranges.append((pos, pos + len(chunk.text)))
            search_from = pos + 1
        else:
            raw_ranges.append((search_from, search_from + len(chunk.text)))

    # Build display ranges with overlap zones
    ranges = []
    for i, (s, e) in enumerate(raw_ranges):
        ranges.append((s, e, i))
        if i > 0 and overlap > 0:
            prev_s, prev_e = raw_ranges[i - 1]
            prev_text = text[prev_s:prev_e]
            prev_tokens = _enc.encode(prev_text)
            if len(prev_tokens) >= overlap:
                ov_text = _enc.decode(prev_tokens[-overlap:])
                ov_start = prev_e - len(ov_text)
                ranges.append((ov_start, prev_e, i))

    return ranges


# ── HTML rendering ─────────────────────────────────────────────────────

def render_text_map(text, ranges):
    """Render text with character-level chunk coloring."""
    n = len(text)

    # Build character ownership map
    owners = [set() for _ in range(n)]
    for s, e, idx in ranges:
        for i in range(max(0, s), min(e, n)):
            owners[i].add(idx)

    # Group consecutive chars with same ownership
    spans = []
    if n > 0:
        i = 0
        while i < n:
            own = frozenset(owners[i])
            j = i + 1
            while j < n and frozenset(owners[j]) == own:
                j += 1
            spans.append((i, j, own))
            i = j

    # Legend
    chunk_ids = sorted(set(r[2] for r in ranges))
    has_overlap = any(len(o) > 1 for _, _, o in spans)

    legend = '<div style="display:flex;flex-wrap:wrap;gap:6px;margin-bottom:12px;">'
    for idx in chunk_ids:
        c = COLORS[idx % len(COLORS)]
        b = BORDERS[idx % len(BORDERS)]
        legend += (
            f'<span style="background:{c};border:1px solid {b};'
            f'padding:2px 10px;border-radius:4px;font-size:12px;'
            f'font-weight:600;">Chunk {idx}</span>'
        )
    if has_overlap:
        legend += (
            '<span style="background:repeating-linear-gradient(45deg,'
            '#ffab91,#ffab91 3px,#90caf9 3px,#90caf9 6px);'
            'padding:2px 10px;border-radius:4px;font-size:12px;'
            'font-weight:600;">Overlap</span>'
        )
    legend += '</div>'

    stats = (
        f'<div style="font-size:12px;color:#666;margin-bottom:8px;">'
        f'{len(chunk_ids)} chunks | {count_tokens(text):,} tokens</div>'
    )

    # Render spans
    body = ''
    for start, end, own in spans:
        raw = text[start:end]
        esc = (raw.replace('&', '&amp;').replace('<', '&lt;')
               .replace('>', '&gt;').replace('\n', '<br>'))

        if not own:
            body += f'<span style="color:#ccc;">{esc}</span>'
        elif len(own) == 1:
            idx = list(own)[0]
            c = COLORS[idx % len(COLORS)]
            body += f'<span style="background:{c};padding:1px 0;">{esc}</span>'
        else:
            ix = sorted(own)
            c1 = COLORS[ix[0] % len(COLORS)]
            c2 = COLORS[ix[1] % len(COLORS)]
            bg = (f"repeating-linear-gradient(45deg,"
                  f"{c1},{c1} 3px,{c2} 3px,{c2} 6px)")
            body += (
                f'<span style="background:{bg};padding:1px 0;font-weight:bold;"'
                f' title="Overlap: chunks {ix[0]} & {ix[1]}">{esc}</span>'
            )

    container = (
        '<div style="font-family:Menlo,monospace;font-size:13px;line-height:1.9;'
        'padding:20px;border:1px solid #e0e0e0;border-radius:8px;'
        'max-height:500px;overflow-y:auto;background:#fafafa;'
        'white-space:pre-wrap;word-wrap:break-word;">'
        f'{body}</div>'
    )

    return legend + stats + container


# ── Chart helper ───────────────────────────────────────────────────────

def make_size_chart(text, strategy, chunk_size, overlap, title="Chunk Sizes"):
    """Plotly bar chart of actual chunk token counts."""
    if strategy == "fixed":
        chunks = chunk_fixed(text, source="demo", chunk_size=chunk_size, overlap=overlap)
    else:
        chunks = chunk_document(text, source="demo", chunk_size=chunk_size, overlap=overlap)

    sizes = [c.metadata.get('token_count', count_tokens(c.text)) for c in chunks]
    if not sizes:
        return go.Figure()

    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=[f"Chunk {i}" for i in range(len(sizes))],
        y=sizes,
        marker_color=[COLORS[i % len(COLORS)] for i in range(len(sizes))],
        marker_line_color=[BORDERS[i % len(BORDERS)] for i in range(len(sizes))],
        marker_line_width=1.5,
        text=sizes,
        textposition='outside',
    ))
    fig.add_hline(
        y=chunk_size, line_dash="dash", line_color="#e53935", line_width=1.5,
        annotation_text=f"Target: {chunk_size}",
        annotation_position="top right",
    )
    fig.update_layout(
        title=title,
        yaxis_title="Tokens",
        height=280,
        margin=dict(t=50, b=40, l=50, r=20),
        plot_bgcolor='white',
        yaxis=dict(gridcolor='#eee'),
    )
    return fig


print("Visualization functions loaded.")

---

## 1. Fixed-Size Chunking: Every N Tokens

Fixed-size chunking is the simplest strategy: cut the text every `chunk_size` tokens, with `overlap` tokens carried over from the previous chunk.

**Watch for:**
- What happens at chunk boundaries -- does a word or sentence get split?
- How overlap changes the coloring at boundaries (striped = overlap zone)
- What happens when you increase overlap -- more chunks? More cost?

In [ ]:
def show_fixed(chunk_size, overlap, doc_name):
    text = documents[doc_name]
    if overlap >= chunk_size:
        print(f"Overlap ({overlap}) must be less than chunk_size ({chunk_size}).")
        return

    ranges = compute_fixed_ranges(text, chunk_size, overlap)
    display(HTML(render_text_map(text, ranges)))

    fig = make_size_chart(text, "fixed", chunk_size, overlap,
                          title="Fixed-Size Chunk Lengths")
    fig.show()

cs = widgets.IntSlider(value=75, min=25, max=300, step=25,
                       description='Chunk Size (tokens):',
                       style={'description_width': '150px'})
ov = widgets.IntSlider(value=12, min=0, max=75, step=5,
                       description='Overlap (tokens):',
                       style={'description_width': '150px'})
doc = widgets.Dropdown(options=list(documents.keys()),
                       description='Document:',
                       style={'description_width': '100px'})

out = widgets.interactive_output(show_fixed,
    {'chunk_size': cs, 'overlap': ov, 'doc_name': doc})
display(widgets.VBox([widgets.HBox([cs, ov]), doc, out]))

---

## 2. Paragraph-Aware Chunking: Respecting Boundaries

Paragraph-aware chunking splits on `\n\n` boundaries, then merges small paragraphs until reaching `chunk_size`. This keeps complete thoughts together.

**Watch for:**
- How chunk sizes vary (unlike fixed, they are NOT uniform)
- Where boundaries land -- end of paragraph vs mid-sentence
- Gray text between chunks -- those are paragraph separators the chunker discards
- Overlap zones that bridge back into the previous chunk's territory

In [ ]:
def show_paragraph(chunk_size, overlap, doc_name):
    text = documents[doc_name]
    if overlap >= chunk_size:
        print(f"Overlap ({overlap}) must be less than chunk_size ({chunk_size}).")
        return

    ranges = compute_paragraph_ranges(text, chunk_size, overlap)
    if not ranges:
        print("No chunks produced. Try a different document or smaller chunk_size.")
        return

    display(HTML(render_text_map(text, ranges)))

    fig = make_size_chart(text, "paragraph", chunk_size, overlap,
                          title="Paragraph-Aware Chunk Lengths")
    fig.show()

cs2 = widgets.IntSlider(value=75, min=25, max=300, step=25,
                        description='Chunk Size (tokens):',
                        style={'description_width': '150px'})
ov2 = widgets.IntSlider(value=12, min=0, max=75, step=5,
                        description='Overlap (tokens):',
                        style={'description_width': '150px'})
doc2 = widgets.Dropdown(options=list(documents.keys()),
                        description='Document:',
                        style={'description_width': '100px'})

out2 = widgets.interactive_output(show_paragraph,
    {'chunk_size': cs2, 'overlap': ov2, 'doc_name': doc2})
display(widgets.VBox([widgets.HBox([cs2, ov2]), doc2, out2]))

---

## 3. Side-by-Side: Same Text, Two Strategies

Before you run the next cell, **predict:**

1. Which strategy will produce more chunks -- fixed or paragraph-aware?
2. Which will have more consistent chunk sizes?
3. Which will have cleaner boundaries (ending at sentence or paragraph breaks)?

Write your predictions down, then run the cell and compare.

In [ ]:
def show_comparison(chunk_size, overlap, doc_name):
    text = documents[doc_name]
    if overlap >= chunk_size:
        print(f"Overlap ({overlap}) must be less than chunk_size ({chunk_size}).")
        return

    fixed_ranges = compute_fixed_ranges(text, chunk_size, overlap)
    para_ranges = compute_paragraph_ranges(text, chunk_size, overlap)

    display(HTML('<h3 style="color:#e65100;">Fixed-Size Strategy</h3>'))
    display(HTML(render_text_map(text, fixed_ranges)))

    display(HTML('<h3 style="color:#1565c0;margin-top:24px;">Paragraph-Aware Strategy</h3>'))
    display(HTML(render_text_map(text, para_ranges)))

    # Comparative bar chart
    fc = chunk_fixed(text, source="demo", chunk_size=chunk_size, overlap=overlap)
    pc = chunk_document(text, source="demo", chunk_size=chunk_size, overlap=overlap)

    fs = [c.metadata.get('token_count', count_tokens(c.text)) for c in fc]
    ps = [c.metadata.get('token_count', count_tokens(c.text)) for c in pc]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        name="Fixed", x=[f"Chunk {i}" for i in range(len(fs))],
        y=fs, marker_color="#ffab91",
    ))
    fig.add_trace(go.Bar(
        name="Paragraph", x=[f"Chunk {i}" for i in range(len(ps))],
        y=ps, marker_color="#90caf9",
    ))
    fig.add_hline(y=chunk_size, line_dash="dash", line_color="#e53935", line_width=1)
    fig.update_layout(
        title="Chunk Size Comparison",
        barmode='group', yaxis_title="Tokens",
        height=300, margin=dict(t=50, b=40),
        plot_bgcolor='white', yaxis=dict(gridcolor='#eee'),
    )
    fig.show()

    # Boundary quality
    def bq(chunks_list):
        clean = sum(1 for c in chunks_list[:-1]
                    if c.text.rstrip().endswith(('.', '!', '?', ':')))
        total = max(len(chunks_list) - 1, 1)
        return clean, total

    fclean, ftotal = bq(fc)
    pclean, ptotal = bq(pc)

    display(HTML(f'''
    <table style="border-collapse:collapse;margin-top:16px;font-size:14px;">
    <tr style="background:#f5f5f5;">
      <th style="padding:8px 16px;text-align:left;"></th>
      <th style="padding:8px 16px;">Fixed</th>
      <th style="padding:8px 16px;">Paragraph</th>
    </tr>
    <tr>
      <td style="padding:6px 16px;">Chunk count</td>
      <td style="padding:6px 16px;text-align:center;">{len(fs)}</td>
      <td style="padding:6px 16px;text-align:center;">{len(ps)}</td>
    </tr>
    <tr style="background:#fafafa;">
      <td style="padding:6px 16px;">Avg size</td>
      <td style="padding:6px 16px;text-align:center;">{sum(fs)//max(len(fs),1)}</td>
      <td style="padding:6px 16px;text-align:center;">{sum(ps)//max(len(ps),1)}</td>
    </tr>
    <tr>
      <td style="padding:6px 16px;">Size range</td>
      <td style="padding:6px 16px;text-align:center;">{min(fs)}-{max(fs)}</td>
      <td style="padding:6px 16px;text-align:center;">{min(ps)}-{max(ps)}</td>
    </tr>
    <tr style="background:#fafafa;">
      <td style="padding:6px 16px;">Clean boundaries</td>
      <td style="padding:6px 16px;text-align:center;">{fclean}/{ftotal} ({fclean*100//max(ftotal,1)}%)</td>
      <td style="padding:6px 16px;text-align:center;">{pclean}/{ptotal} ({pclean*100//max(ptotal,1)}%)</td>
    </tr>
    </table>
    '''))

cs3 = widgets.IntSlider(value=75, min=25, max=300, step=25,
                        description='Chunk Size (tokens):',
                        style={'description_width': '150px'})
ov3 = widgets.IntSlider(value=12, min=0, max=75, step=5,
                        description='Overlap (tokens):',
                        style={'description_width': '150px'})
doc3 = widgets.Dropdown(options=list(documents.keys()),
                        description='Document:',
                        style={'description_width': '100px'})

out3 = widgets.interactive_output(show_comparison,
    {'chunk_size': cs3, 'overlap': ov3, 'doc_name': doc3})
display(widgets.VBox([widgets.HBox([cs3, ov3]), doc3, out3]))

---

## 4. Overlap Deep-Dive

Overlap means the end of one chunk is repeated at the start of the next. This is insurance against information lost at boundaries.

The cell below zooms into a single boundary: it shows the previous chunk and the next chunk, with the shared overlap text highlighted in stripes. Use the slider to adjust how much overlap is applied and watch the shared zone grow or shrink.

In [ ]:
def show_overlap_zoom(chunk_size, overlap, boundary_idx, doc_name):
    text = documents[doc_name]
    if overlap >= chunk_size:
        print("Overlap must be less than chunk_size.")
        return

    chunks = chunk_fixed(text, source="demo", chunk_size=chunk_size, overlap=overlap)

    if len(chunks) < 2:
        print("Only 1 chunk -- no boundaries to inspect. Reduce chunk_size.")
        return

    idx = min(boundary_idx, len(chunks) - 2)
    chunk_a = chunks[idx]
    chunk_b = chunks[idx + 1]

    def esc(s):
        return (s.replace('&', '&amp;').replace('<', '&lt;')
                 .replace('>', '&gt;').replace('\n', '<br>'))

    ca = COLORS[idx % len(COLORS)]
    cb = COLORS[(idx + 1) % len(COLORS)]
    stripe = (f"repeating-linear-gradient(45deg,"
              f"{ca},{ca} 3px,{cb} 3px,{cb} 6px)")

    # Token-based overlap splitting
    tokens_a = _enc.encode(chunk_a.text)
    tokens_b = _enc.encode(chunk_b.text)

    if overlap > 0 and overlap <= len(tokens_a):
        unique_a = _enc.decode(tokens_a[:-overlap])
        shared_a = _enc.decode(tokens_a[-overlap:])
        html_a = (
            f'<span style="background:{ca};padding:1px 0;">{esc(unique_a)}</span>'
            f'<span style="background:{stripe};padding:1px 0;font-weight:bold;">'
            f'{esc(shared_a)}</span>'
        )
    else:
        html_a = f'<span style="background:{ca};padding:1px 0;">{esc(chunk_a.text)}</span>'

    if overlap > 0 and overlap <= len(tokens_b):
        shared_b = _enc.decode(tokens_b[:overlap])
        unique_b = _enc.decode(tokens_b[overlap:])
        html_b = (
            f'<span style="background:{stripe};padding:1px 0;font-weight:bold;">'
            f'{esc(shared_b)}</span>'
            f'<span style="background:{cb};padding:1px 0;">{esc(unique_b)}</span>'
        )
    else:
        html_b = f'<span style="background:{cb};padding:1px 0;">{esc(chunk_b.text)}</span>'

    box = ("font-family:Menlo,monospace;font-size:13px;line-height:1.9;"
           "padding:16px;border:1px solid #e0e0e0;border-radius:8px;"
           "background:#fafafa;white-space:pre-wrap;word-wrap:break-word;"
           "max-height:300px;overflow-y:auto;")

    n_tokens_a = len(tokens_a)
    n_tokens_b = len(tokens_b)
    overlap_tokens = min(overlap, len(tokens_a)) if overlap > 0 else 0

    separator = (f'--- {overlap_tokens} tokens shared ---' if overlap > 0
                 else '--- no overlap ---')

    display(HTML(f'''
    <div style="margin-bottom:8px;font-weight:600;">
      Chunk {idx} ({n_tokens_a} tokens) -- overlap zone at END
    </div>
    <div style="{box}">{html_a}</div>
    <div style="text-align:center;padding:8px;font-size:16px;color:#666;">
      {separator}
    </div>
    <div style="margin-bottom:8px;font-weight:600;">
      Chunk {idx + 1} ({n_tokens_b} tokens) -- overlap zone at START
    </div>
    <div style="{box}">{html_b}</div>
    '''))

cs4 = widgets.IntSlider(value=75, min=25, max=300, step=25,
                        description='Chunk Size (tokens):',
                        style={'description_width': '150px'})
ov4 = widgets.IntSlider(value=20, min=0, max=75, step=5,
                        description='Overlap (tokens):',
                        style={'description_width': '150px'})
bi4 = widgets.IntSlider(value=0, min=0, max=20, step=1,
                        description='Boundary #:',
                        style={'description_width': '100px'})
doc4 = widgets.Dropdown(options=list(documents.keys()),
                        description='Document:',
                        style={'description_width': '100px'})

out4 = widgets.interactive_output(show_overlap_zoom,
    {'chunk_size': cs4, 'overlap': ov4, 'boundary_idx': bi4, 'doc_name': doc4})
display(widgets.VBox([widgets.HBox([cs4, ov4, bi4]), doc4, out4]))

---

## What Did You See?

Before heading back to the main notebook, reflect on these observations:

- [ ] **Boundary cuts:** Where did fixed-size chunking split mid-word or mid-sentence?
- [ ] **Overlap insurance:** How did increasing overlap change the striped zones? Did it always help?
- [ ] **Paragraph-aware wins:** On which documents did paragraph-aware produce noticeably cleaner boundaries?
- [ ] **Uneven sizes:** Paragraph-aware chunks vary in size -- is that a problem or a feature?
- [ ] **Cost tradeoff:** More overlap = more chunks = more embedding cost. Where is the sweet spot?

---

**Head back to `session_2_2.ipynb`** to continue with the Chunk Destruction Demo and the full ingestion pipeline.